# atml-pa3-t2-smoke

Stage: `smoke`

Self-contained ATML PA3 Task 2 runner. Source is embedded in the notebook. Artifacts go to `/kaggle/working/results`. Internet is required to download COCO/VQAv2 files and the SigLIP and Qwen weights.


In [ ]:
import os, sys, shutil, pathlib, zipfile, io, base64
STAGE = 'smoke'
dst = pathlib.Path('/kaggle/working/task2')
if dst.exists():
    shutil.rmtree(dst)
dst.mkdir(parents=True)
print('Extracting embedded code zip')
CODE_B64 = '''...'''
raw = base64.b64decode(CODE_B64)
with zipfile.ZipFile(io.BytesIO(raw)) as zf:
    zf.extractall(dst)
assert (dst / 'src').is_dir() and (dst / 'configs').is_dir(), list(dst.iterdir())
vlm_src = (dst / 'src' / 'models' / 'vlm.py').read_text(encoding='utf-8')
siglip_src = (dst / 'src' / 'models' / 'siglip.py').read_text(encoding='utf-8')
print('HAS_RUN_LM_FORWARD', 'def run_lm_forward' in vlm_src)
print('HAS_THINKING_OFF', 'enable_thinking' in vlm_src and 'False' in vlm_src)
print('LAYERNORM_ONCE', 'exactly once' in siglip_src)
os.chdir(dst)
sys.path.insert(0, str(dst))
print('CODE_ROOT', dst)
print('STAGE', STAGE)


In [ ]:
import torch, subprocess, sys
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), 'VRAM_GiB', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'])
pkgs = [
    'transformers>=4.51.0', 'peft==0.14.0', 'accelerate', 'sentencepiece', 'protobuf',
    'pyyaml', 'safetensors', 'pillow', 'tqdm', 'pandas', 'matplotlib', 'huggingface_hub',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + pkgs)


In [ ]:
import os, sys, subprocess, pathlib, json, time, traceback, shutil
os.environ['ATML_ARTIFACTS_ROOT'] = '/kaggle/working/results'
os.environ['HF_HOME'] = '/tmp/hf_home'
os.environ['HF_HUB_CACHE'] = '/tmp/hf_home'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_home'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if STAGE == 'smoke':
    os.environ['ATML_SMOKE'] = '1'
pathlib.Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)
t0 = time.time()
cmd = [sys.executable, '-m', 'src.run_stage', '--stage', STAGE, '--artifacts-root', '/kaggle/working/results']
print('Running', cmd, flush=True)
try:
    rc = subprocess.call(cmd)
except Exception:
    pathlib.Path('/kaggle/working/results/failure.txt').write_text(traceback.format_exc())
    raise
print('RC', rc, 'elapsed_min', round((time.time() - t0) / 60, 2))
status = {'stage': STAGE, 'returncode': rc, 'elapsed_sec': time.time() - t0}
pathlib.Path('/kaggle/working/results/stage_status.json').write_text(json.dumps(status, indent=2))
shutil.rmtree('/kaggle/working/task2', ignore_errors=True)
shutil.rmtree('/tmp/hf_home', ignore_errors=True)
if rc != 0:
    raise SystemExit(rc)


In [ ]:
import pathlib
root = pathlib.Path('/kaggle/working/results')
files = sorted(str(p.relative_to(root)) for p in root.rglob('*') if p.is_file())
print('artifact_files', len(files))
for path in files[:160]:
    print(path)
status = root / 'stage_status.json'
print(status.read_text() if status.exists() else 'no status')
